<a href="https://colab.research.google.com/github/JulTob/Data/blob/main/Visualizacion/U4_2_2_T%C3%A9cnicas_de_visualizaci%C3%B3n_Geoespacial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# U4.2.2. Visualización Geoespacial Avanzada II: Dinámica y Comparativa

**Asignatura:** Técnicas de Visualización (Grado en Ciencia de Datos - UMH)  
**Unidad:** U4 (Datos Geográficos) y U5 (Interactividad)  
**Herramientas:** Python, Pandas, Folium, geopandas, numpy.

---

### Objetivos de Aprendizaje
En la sesión anterior aprendimos a pintar puntos estáticos. Pero el mundo **no es estático**. Los datos cambian con el tiempo, tienen fronteras políticas y flujos de movimiento. Hoy aprenderemos a:

1.  **Analizar Territorios (Coropletas):** Cómo unir datos estadísticos (`Excel/Pandas`) con fronteras geográficas (Polígonos).
2.  **Visualizar el Tiempo (Time Maps):** Representar la evolución temporal de un fenómeno (tráfico, virus, usuarios).
3.  **Comparar Escenarios (DualMap):** La técnica definitiva para A/B Testing geográfico. un A/B Test consiste en comparar dos versiones de algo para ver cuál funciona mejor.
4.  **Dibujar Flujos (AntPath):** Representar la dirección y velocidad de una ruta.

In [21]:
# 1. IMPORTACIÓN DE LIBRERÍAS
# Necesitamos 'geopandas' para manejar las fronteras de los países.
%pip install folium pandas geopandas numpy

import folium
from folium import plugins
import pandas as pd
import geopandas as gpd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

print("Entorno listo para Smart City Analytics.")

Entorno listo para Smart City Analytics.


---
## Parte 1: El Nivel Macro (Coropletas)

**El Problema:** Somos analistas de la Unión Europea. Tenemos un Excel con el "Índice de Innovación 2025" de cada país, pero en una tabla es difícil ver si existe una brecha Norte-Sur o Este-Oeste.

**La Técnica:** **Mapa de Coropletas (Choropleth)**.
* **¿Qué es?**: Colorear regiones (polígonos) según un valor numérico.
* **Concepto clave**: El `key_on`. Es el "pegamento" que une el nombre del país en tu Excel con el nombre del país en el archivo del mapa (GeoJSON).

In [22]:
# --- PASO 1: OBTENER GEOMETRÍA (El Mapa Mudo) ---
# GeoPandas trae un mapa básico del mundo incorporado.
try:
    print("Descargando mapa desde Natural Earth (versión moderna)...")
    url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
    print("Descargado")
    world = gpd.read_file(url)

    # Ajuste de compatibilidad: Convertimos columnas a minúsculas
    # (El mapa online trae 'CONTINENT' en mayúsculas, tu código espera 'continent')
    world.columns = world.columns.str.lower()

    # Filtramos solo Europa para centrar el tiro
    # Aseguramos que 'continent' existe o usamos 'region_un' como fallback si fuera necesario
    if 'continent' in world.columns:
        europa_geo = world[world.continent == 'Europe'].copy()
    else:
        # Fallback por si el dataset descargado tiene otra estructura
        print("Advertencia: Buscando columna de continente alternativa...")
        europa_geo = world[world['min_zoom'] < 20].copy() # Placeholder si falla la estructura
except:
    print("Error cargando GeoPandas. Verifica tu conexión.")


Descargando mapa desde Natural Earth (versión moderna)...
Descargado


In [23]:
# --- PASO 2: SIMULAR DATOS DE NEGOCIO ---
# Vamos a crear una columna ficticia: 'Innovation_Index' (0 a 100)
np.random.seed(123)
europa_geo['Innovation_Index'] = np.random.randint(40, 98, size=len(europa_geo))

print("Datos preparados. Observa que tenemos la columna 'geometry' (Polígonos) y el 'Index'.")
display(europa_geo[['name', 'pop_est', 'Innovation_Index', 'geometry']].head())

Datos preparados. Observa que tenemos la columna 'geometry' (Polígonos) y el 'Index'.


,name,pop_est,Innovation_Index,geometry
18,Russia,144373535.0,85,"MULTIPOLYGON (((178.7253 71.0988, 180 71.51571..."
21,Norway,5347896.0,42,"MULTIPOLYGON (((15.14282 79.67431, 15.52255 80..."
43,France,67059887.0,68,"MULTIPOLYGON (((-51.6578 4.15623, -52.24934 3...."
110,Sweden,10285453.0,74,"POLYGON ((11.02737 58.85615, 11.46827 59.43239..."
111,Belarus,9466856.0,78,"POLYGON ((28.17671 56.16913, 29.22951 55.91834..."


In [24]:
# --- PASO 3: VISUALIZACIÓN ---

m_coropleta = folium.Map(location=[50, 10], zoom_start=4, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=europa_geo,           # 1. El Mapa (Polígonos)
    name='Innovación 2025',
    data=europa_geo,               # 2. Los Datos
    columns=['name', 'Innovation_Index'], # [Columna Clave, Columna Valor]
    key_on='feature.properties.name',     # 3. El Pegamento (Ruta en el GeoJSON)
    fill_color='YlGnBu',           # Paleta: Yellow-Green-Blue
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Índice de Innovación'
).add_to(m_coropleta)

# Añadimos tooltip para ver los datos al pasar el ratón
folium.GeoJson(
    europa_geo,
    style_function=lambda x: {'fillColor': 'gold', 'color': 'goldenrod'},
    tooltip=folium.GeoJsonTooltip(
        fields=['name', 'Innovation_Index'],
        aliases=['País:', 'Score:']
    )
).add_to(m_coropleta)

m_coropleta.save('U4.2.2_Coropleta_Europa.html')

**Vamos a arreglar los defectos visuales del mapa anterior añadiendo:**
1. Interactive Highlight: El país se ilumina al pasar el ratón (útil para países pequeños).
2. Tooltip Completo: Muestra Nombre + Valor exacto.

In [25]:
# --- VISUALIZACIÓN MEJORADA (INTERACTIVA) ---
# Vamos a arreglar los defectos visuales del mapa anterior añadiendo:
# 1. Interactive Highlight: El país se ilumina al pasar el ratón (útil para países pequeños).
# 2. Tooltip Completo: Muestra Nombre + Valor exacto.

m_coropleta_pro = folium.Map(location=[50, 10], zoom_start=4, tiles='CartoDB positron')

# Capa Base (Pinta los colores)
choropleth = folium.Choropleth(
    geo_data=europa_geo,
    name='Innovación 2025',
    data=europa_geo,
    columns=['name', 'Innovation_Index'],
    key_on='feature.properties.name',
    fill_color='YlGnBu',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Índice de Innovación (0-100)'
).add_to(m_coropleta_pro)

# Capa de Interacción (Bordes brillantes + Tooltip)
# Nota: Usamos el GeoJSON original para añadir funcionalidad de "Highlight"
style_function = lambda x: {'fillColor': '#ffffff', 'color':'#000000', 'fillOpacity': 0.1, 'weight': 0.1}
highlight_function = lambda x: {'fillColor': '#000000', 'color':'#000000', 'fillOpacity': 0.50, 'weight': 0.1}

folium.GeoJson(
    europa_geo,
    style_function=lambda x: {'fillColor': '#00000000', 'color': '#00000000'}, # Invisible por defecto
    highlight_function=lambda x: {'weight': 3, 'color': 'white'}, # Borde blanco al pasar ratón
    tooltip=folium.GeoJsonTooltip(
        fields=['name', 'Innovation_Index'],
        aliases=['País:', 'Score:'],
        style="background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;"
    )
).add_to(m_coropleta_pro)

folium.LayerControl().add_to(m_coropleta_pro)
m_coropleta_pro.save('U4.2.2_Coropleta_Europa_Interactiva.html')

### Parada para Reflexionar: ¿Qué nos dice realmente este mapa?

Acabamos de generar un mapa técnicamente perfecto, pero analíticamente peligroso. Analicémoslo en tres niveles:

**1. Nivel Técnico (El Éxito)**
* El mapa funciona porque el **`key_on`** ha hecho su trabajo: ha unido el nombre "Spain" de nuestro Excel (simulado) con el polígono "Spain" del GeoJSON. Si los nombres no coincidieran, el país saldría gris (sin datos).

**2. Nivel Visual (El Sesgo de Área)**
* Mirad el mapa. **¿Hacia dónde se van vuestros ojos?** Probablemente hacia el Este (Rusia, Ucrania) o el Norte (Escandinavia).
* Esto es el **Area Bias**. En un mapa de coropletas, los países grandes dominan visualmente. Si un país pequeño pero importante (como Suiza o Países Bajos) tuviera un valor crítico, apenas se vería frente a una mancha gigante de color azul.
* *Solución:* Usar tooltips interactivos (como hemos hecho arriba) o complementar el mapa con un gráfico de barras.

**3. Nivel Ético (La Verdad de los Datos)**
* Este mapa parece muy profesional y autoritario. Un directivo podría decidir invertir en base a estos colores.
* Sin embargo, recordad que hemos usado `np.random`. **Los datos son inventados**.
* **Lección:** Una visualización bonita no hace que el dato sea veraz. Como científicos de datos, vuestra prioridad es validar la fuente antes de pintar el mapa. No dejéis que la estética disfrace la falta de rigor.

**El mapa de colores (coropletas) miente porque Rusia parece más importante que Suiza solo por ser grande. Para arreglarlo, vamos a ignorar el tamaño del país. Usaremos GeoPandas para buscar el punto central (centroide) de cada nación y pondremos allí una burbuja. Si el país innova mucho, la burbuja será grande, aunque el país sea una isla minúscula**

### Solución al Sesgo: Mapas de Burbujas (Centroides)

Para evitar que el tamaño del país distorsione la percepción del dato, la solución técnica es **desvincular la variable del polígono**.

Usaremos la geometría para calcular el **Centroide** (el punto central matemático del país) y colocaremos ahí una burbuja cuyo tamaño dependa exclusivamente del `Innovation_Index`.

**GeoPandas** nos permite calcular esto automáticamente con `.centroid`.

In [26]:
# 1. Calculamos los centroides (puntos centrales) de cada país
# Nota: GeoPandas nos avisará de que el cálculo en coordenadas planas no es perfecto,
# pero para visualización web es suficiente.
# 1. Calculamos Centroides (igual que antes)
europa_geo['centroide'] = europa_geo.geometry.centroid

# 2. CAMBIO DE ESTILO: Usamos 'CartoDB positron' (Luz/Gris) para ver bien las fronteras
m_burbujas = folium.Map(location=[50, 15], zoom_start=4, tiles='CartoDB positron')

# 3. Dibujamos las Burbujas
for idx, row in europa_geo.iterrows():
    lat = row['centroide'].y
    lon = row['centroide'].x

    # Escalado del radio
    radio = row['Innovation_Index'] / 3

    folium.CircleMarker(
        location=[lat, lon],
        radius=radio,
        color='#2c3e50',       # Borde: Azul oscuro (para contraste sobre mapa claro)
        weight=1,              # Grosor del borde
        fill=True,
        fill_color='cyan',     # Relleno: Cyan brillante
        fill_opacity=0.6,
        popup=f"<b>{row['name']}</b><br>Index: {row['Innovation_Index']}",
        tooltip=f"{row['name']}: {row['Innovation_Index']}"
    ).add_to(m_burbujas)


print("Observad la diferencia: Ahora los países pequeños con alto índice sí destacan.")
m_burbujas.save('U4.2.2_Burbujas_Europa.html')

Observad la diferencia: Ahora los países pequeños con alto índice sí destacan.


**Conclusión:** Al usar burbujas, países pequeños como **Suiza, Bélgica o Países Bajos** (que en el mapa de coropletas eran invisibles) ahora compiten visualmente en igualdad de condiciones con los gigantes. Esta técnica es mucho más justa para comparar rendimiento.

    ---
    

### Problema detectado
Al calcular el centroide matemático, países como Francia salen mal posicionados. ¿Por qué? Porque GeoPandas hace la media incluyendo sus territorios de ultramar (como la Guayana Francesa en América).

**Solución:** Creamos una función personalizada que seleccione el polígono más grande (el territorio principal) de cada país antes de calcular el centro.

In [36]:
# Función para arreglar países fragmentados (Francia, Noruega, etc.)
def get_main_centroid(geometry):
    # Si es un solo polígono (ej. Polonia), devolvemos su centro directo
    if geometry.geom_type == 'Polygon':
        return geometry.centroid

    # Si son varios trozos (MultiPolygon), buscamos el trozo más grande (European Mainland)
    elif geometry.geom_type == 'MultiPolygon':
        # max() busca el polígono con mayor área
        main_land = max(geometry.geoms, key=lambda g: g.area)
        return main_land.centroid

    else:
        return geometry.centroid

# 1. Aplicamos la corrección
europa_geo['centroide_corregido'] = europa_geo.geometry.apply(get_main_centroid)

# Ensure 'Innovation_Index' exists (in case previous cells weren't run or state was lost)
import numpy as np # Ensure numpy is imported if not already
if 'Innovation_Index' not in europa_geo.columns:
    np.random.seed(123) # Use the same seed for consistency
    europa_geo['Innovation_Index'] = np.random.randint(40, 98, size=len(europa_geo))

# 2. Creamos el mapa (Estilo Positron para ver fronteras)
m_burbujas = folium.Map(location=[50, 15], zoom_start=4, tiles='CartoDB positron')

# 3. Dibujamos las Burbujas usando el CENTROIDE CORREGIDO
for idx, row in europa_geo.iterrows():
    # Usamos la nueva columna corregida
    lat = row['centroide_corregido'].y
    lon = row['centroide_corregido'].x

    radio = row['Innovation_Index'] / 3

    folium.CircleMarker(
        location=[lat, lon],
        radius=radio,
        color='#2c3e50',
        weight=1,
        fill=True,
        fill_color='cyan',
        fill_opacity=0.6,
        popup=f"<b>{row['name']}</b><br>Index: {row['Innovation_Index']}",
        tooltip=f"{row['name']}: {row['Innovation_Index']}"
    ).add_to(m_burbujas)

print("Corrección aplicada: Ahora Francia (y Noruega) tienen la burbuja en su sitio.")
m_burbujas.save('U4.2.2_Burbujas_Europa_Corregido.html')

Corrección aplicada: Ahora Francia (y Noruega) tienen la burbuja en su sitio.



---

## Parte 2: El Tiempo (HeatMapWithTime)

Hasta ahora, nuestros mapas eran **fotografías**: una imagen estática de un momento concreto. Pero la realidad es dinámica.

**El Problema:** Una startup de alquiler de patinetes eléctricos quiere ver cómo se mueven sus usuarios en Madrid a lo largo de 24 horas. ¿Se concentran en el centro por la mañana? ¿Se dispersan por la noche?

* Un mapa estático os dice: *"Hay muchos patinetes en el centro"*.
* Un mapa temporal os dice: *"Los patinetes entran al centro a las 09:00 y salen a las 18:00"*.

**La Técnica:** **HeatMapWithTime**
* **Dato necesario:** Una lista tridimensional. `[ [Puntos Hora 0], [Puntos Hora 1], ... ]`.
* **Valor:** Permite entender patrones de comportamiento dinámico.

### La Metáfora del Cine
Para usar `HeatMapWithTime`, no penséis en tablas de datos. Pensad que sois **directores de cine**. Una película no es más que una pila de fotos mostradas muy rápido. `Folium` necesita exactamente eso:

1.  **La Película (La Lista Principal):** Contiene todo el día.
2.  **El Fotograma (La Sub-lista):** Representa UNA hora concreta.
3.  **Los Actores (Las Coordenadas):** Dónde está cada patinete en ESE fotograma.

### La Estructura de Datos (Lista de listas de listas)

```python
datos_pelicula = [
    # --- FOTOGRAMA 1: Las 08:00 AM ---
    [ [40.4, -3.7], [40.41, -3.71], [40.39, -3.69] ],

    # --- FOTOGRAMA 2: Las 09:00 AM (Los puntos han cambiado de sitio) ---
    [ [40.42, -3.72], [40.43, -3.70], [40.40, -3.68] ],
    
    # ... resto del día ...
]


In [28]:
# --- SIMULACIÓN: UN DÍA EN MADRID (VERSIÓN ARREGLADA) ---
import numpy as np

lat_centro, lon_centro = 40.4168, -3.7038 # Puerta del Sol

# Configuración
horas = 24
patinetes = 300
datos_pelicula = []   # La lista gigante (El vídeo completo)
indice_horas = []     # Las etiquetas para la barra de reproducción

print("Rodando película... Generando fotogramas...")

for t in range(horas):
    # --- 1. GUIÓN DE LA ESCENA ---
    respiracion = np.sin(np.pi * t / 24)
    dispersion = 0.002 + (0.015 * respiracion)

    # --- 2. POSICIONAR ACTORES ---
    lats = np.random.normal(lat_centro, dispersion, patinetes)
    lons = np.random.normal(lon_centro, dispersion, patinetes)

    # --- 3. MONTAR EL FOTOGRAMA (CORRECCIÓN CRÍTICA AQUÍ) ---
    # Convertimos explícitamente a float() de Python para evitar errores de Numpy
    fotograma_actual = [[float(la), float(lo)] for la, lo in zip(lats, lons)]

    # Añadimos el fotograma al rollo de película
    datos_pelicula.append(fotograma_actual)

    # Guardamos el nombre de la hora
    indice_horas.append(f"{t:02d}:00")

# --- PROYECCIÓN ---
# Cambiamos a mapa claro (Positron) para asegurar que se ve algo
m_tiempo = folium.Map(location=[lat_centro, lon_centro], zoom_start=13, tiles='CartoDB positron')

plugins.HeatMapWithTime(
    datos_pelicula,
    index=indice_horas,
    radius=15,
    auto_play=True,
    max_opacity=0.8,
    name="Actividad Patinetes"
).add_to(m_tiempo)

print("Mapa temporal. Si ves el mapa blanco, dale al PLAY abajo a la izquierda.")
m_tiempo.save('U4.2.2_HeatMapWithTime_Madrid.html')

Rodando película... Generando fotogramas...
Mapa temporal. Si ves el mapa blanco, dale al PLAY abajo a la izquierda.


---
## Parte 3: Comparativa de Escenarios (DualMap)

**El Problema:** El Ayuntamiento quiere evaluar el impacto del tráfico en la calidad del aire. Quiere ver, en la misma pantalla, la situación un **Lunes por la mañana (Atasco)** vs un **Domingo (Tranquilidad)**.

**La Técnica:** **DualMap**.
* **¿Qué es?**: Dos mapas sincronizados. Si haces zoom en uno, el otro te sigue.
* **Uso**: A/B Testing, Antes/Después, Día/Noche.

In [29]:
# Creamos el mapa doble
m_dual = plugins.DualMap(location=[40.44, -3.69], zoom_start=12)

# --- DATOS ESCENARIO A (Izquierda): LUNES (Alta contaminación en Castellana)
# Simulamos puntos concentrados en el norte (zona financiera)
datos_lunes = np.random.normal(loc=[40.45, -3.69], scale=0.01, size=(100, 2)).tolist()

# --- DATOS ESCENARIO B (Derecha): DOMINGO (Baja contaminación, dispersa)
# Simulamos puntos muy dispersos por toda la ciudad
datos_domingo = np.random.normal(loc=[40.42, -3.70], scale=0.04, size=(50, 2)).tolist()

# --- CONFIGURACIÓN MAPA 1 (IZQUIERDA) ---
folium.Marker([40.45, -3.69], popup="Zona Financiera", icon=folium.Icon(color='red', icon='car', prefix='fa')).add_to(m_dual.m1)
plugins.HeatMap(datos_lunes, radius=20, name="Alta Polución", gradient={0.6: 'orange', 1: 'red'}).add_to(m_dual.m1)

# --- CONFIGURACIÓN MAPA 2 (DERECHA) ---
folium.Marker([40.416, -3.703], popup="Zona Centro", icon=folium.Icon(color='green', icon='tree', prefix='fa')).add_to(m_dual.m2)
plugins.HeatMap(datos_domingo, radius=20, name="Baja Polución", gradient={0.6: 'lime', 1: 'green'}).add_to(m_dual.m2)

# Añadimos control de capas para poder encender/apagar
folium.LayerControl().add_to(m_dual)

print("Mapa comparativo generado. Izquierda: Lunes (Rojo) | Derecha: Domingo (Verde)")
m_dual.save('U4.2.2_DualMap_Madrid_Polucion.html')

Mapa comparativo generado. Izquierda: Lunes (Rojo) | Derecha: Domingo (Verde)


In [37]:
import folium
from folium import plugins
import numpy as np

# Creamos el mapa doble
m_dual = plugins.DualMap(location=[40.44, -3.69], zoom_start=12)

# --- DATOS ESCENARIO A (Izquierda): LUNES (Alta contaminación en Castellana)
# Simulamos puntos concentrados en el norte (zona financiera)
datos_lunes = np.random.normal(loc=[40.45, -3.69], scale=0.01, size=(100, 2)).tolist()

# --- DATOS ESCENARIO B (Derecha): DOMINGO (Baja contaminación, dispersa)
# Simulamos puntos muy dispersos por toda la ciudad
datos_domingo = np.random.normal(loc=[40.42, -3.70], scale=0.04, size=(50, 2)).tolist()

# --- CONFIGURACIÓN MAPA 1 (IZQUIERDA) ---
folium.Marker([40.45, -3.69], popup="Zona Financiera", icon=folium.Icon(color='red', icon='car', prefix='fa')).add_to(m_dual.m1)
plugins.HeatMap(datos_lunes, radius=20, name="Alta Polución", gradient={0.6: 'orange', 1: 'red'}).add_to(m_dual.m1)

# --- CONFIGURACIÓN MAPA 2 (DERECHA) ---
folium.Marker([40.416, -3.703], popup="Zona Centro", icon=folium.Icon(color='green', icon='tree', prefix='fa')).add_to(m_dual.m2)
plugins.HeatMap(datos_domingo, radius=20, name="Baja Polución", gradient={0.6: 'lime', 1: 'green'}).add_to(m_dual.m2)

# Añadimos control de capas
folium.LayerControl().add_to(m_dual)

# --- AÑADIDO: TÍTULOS FLOTANTES (HTML INYECTADO) ---
# Creamos dos cajas de texto con HTML/CSS que se quedan fijas en la pantalla
titulos_html = """
<div style="position: fixed; top: 10px; left: 50px; width: 300px; height: 35px;
    z-index:9999; font-size:16px; font-weight:bold; background-color: rgba(255, 255, 255, 0.8);
    padding: 5px; border: 1px solid black; border-radius: 5px; color: black; text-align: center;">
    📅 ESCENARIO A: Lunes (Laborable)
</div>

<div style="position: fixed; top: 10px; right: 50px; width: 300px; height: 35px;
    z-index:9999; font-size:16px; font-weight:bold; background-color: rgba(255, 255, 255, 0.8);
    padding: 5px; border: 1px solid black; border-radius: 5px; color: black; text-align: center;">
    🌳 ESCENARIO B: Domingo (Festivo)
</div>
"""
# Inyectamos este HTML en el mapa
m_dual.get_root().html.add_child(folium.Element(titulos_html))

print("Mapa comparativo generado con títulos.")
m_dual.save('U4.2.2_DualMap_Madrid_Polucion.html')

Mapa comparativo generado con títulos.



---

## Parte 4: Flujos y Rutas (AntPath)

**La Crítica:** Muchos analistas usan líneas estáticas (PolyLine) para representar rutas. Pero en logística y redes, una línea quieta es ambigua: conecta A y B, pero... ¿el flujo va de A hacia B? ¿De B hacia A? ¿Es bidireccional?

**La Técnica**: `AntPath` (Camino de Hormigas). No es solo estética, es información semántica visual:
    * **Direccionalidad**: La animación indica inequívocamente el sentido del flujo (Input -> Output).
    * **Velocidad/Caudal**: Podemos ajustar la velocidad de la animación para representar la velocidad real del tráfico o el ancho de banda de una red.

**El problema**: "La Logística de Ida y Vuelta".
Un camión realiza el circuito Madrid -> Valencia -> Zaragoza -> Madrid. Usaremos `AntPath` para detectar anomalías en la velocidad de cada tramo, ajustando el parámetro `delay` (a mayor `delay`, más lenta la animación):

* Madrid -> Valencia: Camión muy cargado. Esperamos velocidad lenta.
* Valencia -> Zaragoza: Carga parcial. Velocidad media.
* Zaragoza -> Madrid: Retorno vacío. Velocidad alta (vuelta urgente).

**Parámetros clave a observar:**
* **`color`**: Diferencia el estado (Rojo=Carga, Verde=Vacío).
* **`delay`**: Controla la velocidad de las "hormigas". **Cuidado:** Es contraintuitivo.
    * `delay: 2000` = Mucha espera entre pasos = **LENTO**.
    * `delay: 500` = Poca espera entre pasos = **RÁPIDO**.

In [38]:
# Coordenadas Clave
madrid = [40.416, -3.703]
valencia = [39.469, -0.376]
zaragoza = [41.648, -0.889]

# Mapa centrado en el triángulo logístico
m_logistica = folium.Map(location=[40.5, -2.0], zoom_start=7, tiles='CartoDB dark_matter')

# --- TRAMO 1: Madrid -> Valencia (PESADO) ---
plugins.AntPath(
    locations=[madrid, valencia],
    color='red',         # ROJO = Carga Máxima
    weight=6,
    delay=2000,          # 2000ms = LENTO (El camión va lleno)
    dash_array=[10, 50], # Patrón discontinuo amplio (hormigas lentas)
    tooltip="Tramo 1: Carga Pesada (Lento)"
).add_to(m_logistica)

# --- TRAMO 2: Valencia -> Zaragoza (MEDIO) ---
plugins.AntPath(
    locations=[valencia, zaragoza],
    color='orange',      # NARANJA = Carga Media
    weight=5,
    delay=1000,          # 1000ms = VELOCIDAD MEDIA
    dash_array=[15, 30],
    tooltip="Tramo 2: Carga Parcial"
).add_to(m_logistica)

# --- TRAMO 3: Zaragoza -> Madrid (VACÍO/RÁPIDO) ---
plugins.AntPath(
    locations=[zaragoza, madrid],
    color='#00ffcc',     # CYAN = Vacío
    weight=4,
    delay=400,           # 400ms = MUY RÁPIDO (Vuelta urgente)
    dash_array=[20, 10], # Línea casi continua (sensación de velocidad)
    tooltip="Tramo 3: Retorno Vacío (Rápido)"
).add_to(m_logistica)

# Marcadores de las ciudades con iconos semánticos
folium.Marker(madrid, popup="MADRID (Hub)", icon=folium.Icon(icon='city', color='blue', prefix='fa')).add_to(m_logistica)
folium.Marker(valencia, popup="VALENCIA", icon=folium.Icon(icon='anchor', color='blue', prefix='fa')).add_to(m_logistica)
folium.Marker(zaragoza, popup="ZARAGOZA", icon=folium.Icon(icon='industry', color='gray', prefix='fa')).add_to(m_logistica)

print("Visualización generada: Observad cómo la velocidad de las hormigas cambia en cada tramo del triángulo.")
m_logistica.save('U4.2.2_AntPath_Ruta_Logistica.html')


Visualización generada: Observad cómo la velocidad de las hormigas cambia en cada tramo del triángulo.



---

# Práctica de Consolidación: "Data Science en el Mundo Real"

Para terminar la sesión, vamos a resolver 3 casos prácticos. Tenéis el código casi listo, pero faltan las piezas clave que definen la visualización.


### Ejercicio 1: Sudamérica y el Sesgo de Área

**Contexto:** Un fondo de inversión quiere visualizar el **PIB (Simulado)** en Sudamérica.

**Problema:** Brasil es gigante. Si usamos Coropletas, Brasil dominará el mapa visualmente.

**Objetivo:** Crear un mapa de **Burbujas (Centroides)** para comparar economías de forma justa.

**Pasos:**
1.  Filtra el dataset `world` para quedarte con `continent == 'South America'`.
2.  Calcula los centroides.
3.  Completa el código del bucle `for` para dibujar los círculos.

In [32]:
# 1. PREPARACIÓN DE DATOS
# (Asegúrate de tener 'world' cargado de las celdas anteriores)
sudamerica = world[world.continent == 'South America'].copy()
sudamerica['GDP_Simulado'] = np.random.randint(20, 90, size=len(sudamerica))

# --- TU CÓDIGO AQUÍ (Paso 2: Calcula los centroides) ---
# sudamerica['centroide'] = ...

# 2. VISUALIZACIÓN
m_sur = folium.Map(location=[-15, -60], zoom_start=3, tiles='CartoDB positron')

# --- TU CÓDIGO AQUÍ (Paso 3: Completa el bucle) ---
for idx, row in sudamerica.iterrows():
    # lat = ...
    # lon = ...

    folium.CircleMarker(
        location=[lat, lon],
        radius=row['GDP_Simulado'] / 4,  # Escalamos el radio
        color='crimson',
        fill=True,
        tooltip=f"{row['name']}: {row['GDP_Simulado']}"
    ).add_to(m_sur)

m_sur.save('U4.2.2_Burbujas_Sudamerica.html')


---

### Ejercicio 2: El Dron de Reparto (AntPath)

**Contexto:** Estás programando la visualización de un dron de farmacia.

**Ruta:** Hospital -> Paciente -> Hospital.

**Lógica:**
1.  **IDA (Hospital a Paciente):** Va cargado con medicinas. Peso alto = Velocidad Lenta (`delay` alto).
2.  **VUELTA (Paciente a Hospital):** Vuelve vacío. Peso bajo = Velocidad Rápida (`delay` bajo).

**Coordenadas:**
* Hospital: `[40.41, -3.69]`
* Paciente: `[40.45, -3.65]`

In [33]:
import folium
from folium import plugins

# 1. Configuración del Mapa
# Usamos el estilo por defecto (blanco) para asegurar que los colores se vean bien
m_dron = folium.Map(location=[40.43, -3.67], zoom_start=13)

# 2. Coordenadas
hospital = [40.416, -3.690]
paciente = [40.450, -3.650]
punto_giro = [40.460, -3.700] # ¡TRUCO!: Usad este punto para la vuelta para hacer un triángulo

# --- TU CÓDIGO AQUÍ (TRAMO 1: IDA - Directo) ---
# El dron va cargado -> Velocidad LENTA (Delay alto, ej: 2000)
# Ruta: De hospital a paciente
# plugins.AntPath(
#     locations=[ ... , ... ],
#     color='red',
#     weight=5,
#     delay= ...
# ).add_to(m_dron)

# --- TU CÓDIGO AQUÍ (TRAMO 2: VUELTA - Triangular) ---
# El dron vuelve vacío -> Velocidad RÁPIDA (Delay bajo, ej: 500)
# Ruta: De paciente -> a punto_giro -> a hospital
# plugins.AntPath(
#     locations=[ ... , ... , ... ],
#     color='blue',
#     weight=5,
#     delay= ...
# ).add_to(m_dron)

# Añadimos marcadores para referencia
folium.Marker(hospital, popup="Hospital", icon=folium.Icon(color='red', icon='plus', prefix='fa')).add_to(m_dron)
folium.Marker(paciente, popup="Paciente", icon=folium.Icon(color='blue', icon='user', prefix='fa')).add_to(m_dron)

# Truco final: Esto centra el mapa automáticamente para que se vea toda la ruta
m_dron.fit_bounds([hospital, paciente, punto_giro])

m_dron.save('U4.2.2_AntPath_Dron_Salud.html')



---

### Ejercicio 3: El Concierto (HeatMapWithTime)

**Contexto:** Un festival de música. Queremos ver cómo la gente se mueve del **Escenario Principal** a la **Zona de Comida**.

**Datos:** Os doy la estructura del bucle hecha.

**Tarea:**
1.  Entender que la lista `datos_festival` debe ser una **lista de listas de listas**.
2.  Arreglar el error de formato (añadir la conversión a `float` que aprendimos).
3.  Generar el mapa.

In [34]:
escenario = [40.41, -3.71]
comida = [40.42, -3.70]

datos_festival = []
horas = 10
personas = 50

for t in range(horas):
    # Interpolación simple: Los puntos se van moviendo del Escenario a la Comida poco a poco
    factor_movimiento = t / horas
    lat_actual = escenario[0] * (1 - factor_movimiento) + comida[0] * factor_movimiento
    lon_actual = escenario[1] * (1 - factor_movimiento) + comida[1] * factor_movimiento

    # Generamos dispersión aleatoria alrededor del punto actual
    lats = np.random.normal(lat_actual, 0.002, personas)
    lons = np.random.normal(lon_actual, 0.002, personas)

    # --- TU CÓDIGO AQUÍ (Crea el fotograma) ---
    # Recuerda: fotograma = [ [lat, lon], [lat, lon]... ]
    # ¡IMPORTANTE!: Usa float() para evitar errores con Numpy

    fotograma = [] # Rellena esto con un bucle o list comprehension

    # datos_festival.append(fotograma)

# Visualización
m_festival = folium.Map(location=[40.415, -3.705], zoom_start=14, tiles='CartoDB dark_matter')

# plugins.HeatMapWithTime( ... ).add_to(m_festival)
# m_festival